In [42]:
from pathlib import Path
from typing import Annotated, Literal, Type

import instructor
import lancedb
import pandas as pd
from google.generativeai import GenerativeModel
from lancedb.db import DBConnection
from lancedb.embeddings import SentenceTransformerEmbeddings, get_registry
from lancedb.pydantic import LanceModel
from lancedb.pydantic import Vector as LanceVector
from lancedb.table import Table as LanceTable
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import LanceDB
from langchain_core.documents import Document as LCDocument
from pydantic import AfterValidator, BaseModel, Field, create_model

from dreamai.ai import ModelName, count_gpt_tokens, create, system_message, user_message

ask_gemini = instructor.from_gemini(GenerativeModel(model_name=ModelName.GEMINI_FLASH))

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
LANCE_URI = "lance/rag"
# EMS_MODEL = "BAAI/bge-small-en-v1.5"
EMS_MODEL = "hkunlp/instructor-base"
DEVICE = "cuda"
DOCS_LIMIT = 3
CHUNK_SIZE = 800
CHUNK_OVERLAP = 300
SEPARATORS = ["\n\n", "\n", ". "]


def pdf_to_docs(
    pdf_file: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
    separators: list = SEPARATORS,
) -> list[LCDocument]:
    loader = PyPDFLoader(file_path=pdf_file)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=separators,
        keep_separator=False,
    )
    docs = loader.load_and_split(splitter)
    return docs


def create_lance_ems_model(
    name: str = EMS_MODEL, device: str = DEVICE
) -> SentenceTransformerEmbeddings:
    return get_registry().get("sentence-transformers").create(name=name, device=device)


def create_lance_schema(
    name: str,
    ems_model: SentenceTransformerEmbeddings,
    extra_fields: dict | None = None,
) -> Type[LanceModel]:
    extra_fields = extra_fields or {}
    fields = {
        "page_content": (str, ems_model.SourceField()),
        "vector": (LanceVector(dim=ems_model.ndims()), ems_model.VectorField()),  # type: ignore
        **{field: (type(value), ...) for field, value in extra_fields.items()},
    }
    return create_model(name, **fields, __base__=LanceModel)


def add_table(
    db: DBConnection,
    table_name: str,
    data: list[LCDocument],
    ems_model: SentenceTransformerEmbeddings | str,
    schema: Type[LanceModel] | None = None,
    ems_model_device: str = DEVICE,
    overwrite: bool = False,
) -> LanceTable:
    if isinstance(ems_model, str):
        ems_model = create_lance_ems_model(name=ems_model, device=ems_model_device)
    schema = schema or create_lance_schema(
        name="LanceDoc", ems_model=ems_model, extra_fields=data[0].metadata
    )
    table = db.create_table(
        name=table_name, schema=schema, mode="overwrite" if overwrite else "create"
    )
    table.add(data=[{"page_content": d.page_content, **d.metadata} for d in data])
    table.create_fts_index(field_names="page_content", replace=overwrite)  # type: ignore
    return table


def search_lancedb(
    db: DBConnection, table_name: str, query: str, limit: int = DOCS_LIMIT
) -> pd.DataFrame:
    return (
        db.open_table(name=table_name)
        .search(query=query, query_type="hybrid")
        .limit(limit=limit)
        .to_pandas()
    )

In [35]:
book_file = "/media/hamza/data2/stats.pdf"
book_docs = pdf_to_docs(
    pdf_file=book_file,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=SEPARATORS,
)

In [37]:
book_text = "\n".join([p.page_content for p in book_docs if len(p.page_content) > 2])
print(count_gpt_tokens(book_text))

676673


In [39]:
class DataDescription(BaseModel):
    name: str = Field(
        ...,
        description="The name of the data. It should be short and make it obvious what the data is.",
    )
    description: str = Field(
        ..., description="A short description of the data. It should be 1-2 sentences."
    )

In [40]:
desc = ask_gemini.create(
    response_model=DataDescription,
    messages=[
        system_message(
            """\
            You are an expert at condensing text into a short description.
            Given the following text, condense it into a 1-2 sentence description.
            Also, give the data a name.
            """
        ),
        user_message(book_text),
    ],  # type: ignore
)

In [52]:
options = Literal["AI", desc.name]

In [53]:
option = ask_gemini.create(
    response_model=options,
    messages=[
        user_message(f"Routes: AI, {desc.model_dump_json()}."),
        user_message("I want to study about stats. What should I do?"),
    ],  # type: ignore
)

In [54]:
option

'Introductory Statistics Textbook Description'

In [3]:
lance_db = lancedb.connect(LANCE_URI)
ems_model = create_lance_ems_model(name=EMS_MODEL, device=DEVICE)
pdf_docs = pdf_to_docs(pdf_file="resume.pdf")

In [5]:
lance_table = add_table(
    db=lance_db,
    table_name="pdf_docs",
    data=pdf_docs,
    ems_model=ems_model,
    overwrite=True,
)

/home/hamza/dev/dreamai/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
query = "college and university"
search_results = (
    lance_table.search(query=query, query_type="hybrid")
    .where("page = 1", prefilter=True)
    .limit(limit=3)
    .to_pandas()
)

In [7]:
search_results

,page_content,vector,source,page,_relevance_score
0,"EDUCATION \nCloud Computing for Big Data , Po...","[-0.008660806, -0.009674823, -0.001957504, 0.0...",resume.pdf,1,1.000000
1,• Developed Spring Boot applications in micros...,"[-0.036029465, -0.0024531162, 0.0118686315, 0....",resume.pdf,1,0.433564
2,Senior S oftware Engineer Oct 2015 -...,"[-0.0034419084, -0.0051567056, 0.028540252, 0....",resume.pdf,1,0.000000


In [4]:
MAX_COMPONENTS = 5


def validate_sentence_components(
    x: list[str], max_components: int = MAX_COMPONENTS
) -> list[str]:
    return list(set(x[:max_components]))


class SentenceComponents(BaseModel):
    noun: Annotated[list[str], AfterValidator(validate_sentence_components)]
    subject: Annotated[list[str], AfterValidator(validate_sentence_components)]
    object: Annotated[list[str], AfterValidator(validate_sentence_components)]
    verb: Annotated[list[str], AfterValidator(validate_sentence_components)]
    adjective: Annotated[list[str], AfterValidator(validate_sentence_components)]


class StepBackQuestions(BaseModel):
    questions: list[str] = Field(..., min_length=1, max_length=3)

In [36]:
sc = SentenceComponents(
    noun=["football"],
    subject=["Cristiano Ronaldo"],
    object=["Al Nassr"],
    verb=["plays for"],
    adjective=["great"],
)
sc.model_dump()

{'noun': ['football'],
 'subject': ['Cristiano Ronaldo'],
 'object': ['Al Nassr'],
 'verb': ['plays for'],
 'adjective': ['great']}

In [37]:
cr7 = """\
Cristiano Ronaldo dos Santos Aveiro (born 5 February 1985) is a Portuguese professional footballer who plays as a forward for and captains both Saudi Pro League club Al Nassr and the Portugal national team. Widely regarded as one of the greatest players of all time, Ronaldo has won five Ballon d'Or awards,[note 3] a record three UEFA Men's Player of the Year Awards, and four European Golden Shoes, the most by a European player. He has won 33 trophies in his career, including seven league titles, five UEFA Champions Leagues, the UEFA European Championship and the UEFA Nations League. Ronaldo holds the records for most appearances (183), goals (140) and assists (42) in the Champions League, goals in the European Championship (14), international goals (128) and international appearances (205). He is one of the few players to have made over 1,200 professional career appearances, the most by an outfield player, and has scored over 850 official senior career goals for club and country, making him the top goalscorer of all time.
"""
print(cr7.strip())

Cristiano Ronaldo dos Santos Aveiro (born 5 February 1985) is a Portuguese professional footballer who plays as a forward for and captains both Saudi Pro League club Al Nassr and the Portugal national team. Widely regarded as one of the greatest players of all time, Ronaldo has won five Ballon d'Or awards,[note 3] a record three UEFA Men's Player of the Year Awards, and four European Golden Shoes, the most by a European player. He has won 33 trophies in his career, including seven league titles, five UEFA Champions Leagues, the UEFA European Championship and the UEFA Nations League. Ronaldo holds the records for most appearances (183), goals (140) and assists (42) in the Champions League, goals in the European Championship (14), international goals (128) and international appearances (205). He is one of the few players to have made over 1,200 professional career appearances, the most by an outfield player, and has scored over 850 official senior career goals for club and country, makin

In [39]:
res = create(
    messages=[{"role": "user", "content": cr7}],
    system=Path("components_prompt.txt").read_text(),
    model=ModelName.GEMINI_FLASH,
    response_model=SentenceComponents,
)
res

SentenceComponents(noun=['Al Nassr', 'footballer', 'Cristiano Ronaldo dos Santos Aveiro', 'club', 'forward'], subject=['Cristiano Ronaldo dos Santos Aveiro', 'Ronaldo', 'He'], object=['captain', 'awards', 'Awards', 'forward'], verb=['plays', 'captains', 'won', 'regarded'], adjective=['greatest', 'professional', 'national', 'Saudi Pro', 'Portuguese'])

In [40]:
res.model_dump()

{'noun': ['Al Nassr',
  'footballer',
  'Cristiano Ronaldo dos Santos Aveiro',
  'club',
  'forward'],
 'subject': ['Cristiano Ronaldo dos Santos Aveiro', 'Ronaldo', 'He'],
 'object': ['captain', 'awards', 'Awards', 'forward'],
 'verb': ['plays', 'captains', 'won', 'regarded'],
 'adjective': ['greatest',
  'professional',
  'national',
  'Saudi Pro',
  'Portuguese']}